<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)

**Mastering Data-Driven Finance**

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="https://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# Trading Strategies (b)

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
from pylab import mpl, plt
import warnings

In [ ]:
warnings.simplefilter('ignore')
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
np.random.seed(1000)
%config InlineBackend.figure_format = 'svg'

## Linear OLS Regression

### The Data

In [ ]:
raw = pd.read_csv('https://hilpisch.com/tr_eikon_eod_data.csv',
                  index_col=0, parse_dates=True).dropna()

In [ ]:
raw.columns

In [ ]:
symbol = 'EUR='

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data['returns'] = np.log(data / data.shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
data['direction'] = np.sign(data['returns']).astype(int)

In [ ]:
data.head()

In [ ]:
data['returns'].hist(bins=35, figsize=(10, 6));

In [ ]:
lags = 2

In [ ]:
def create_lags(data):
    global cols
    cols = []
    for lag in range(1, lags + 1):
        col = 'lag_{}'.format(lag)
        data[col] = data['returns'].shift(lag)
        cols.append(col)

In [ ]:
create_lags(data)

In [ ]:
data.head()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.plot.scatter(x='lag_1', y='lag_2', c='returns', 
                  cmap='coolwarm', figsize=(10, 6), colorbar=True)
plt.axvline(0, c='r', ls='--')
plt.axhline(0, c='r', ls='--');

### Regression

In [ ]:
from sklearn.linear_model import LinearRegression  

In [ ]:
model = LinearRegression()  

In [ ]:
data['pos_ols_1'] = model.fit(data[cols], data['returns']).predict(data[cols])  

In [ ]:
data['pos_ols_2'] = model.fit(data[cols], data['direction']).predict(data[cols])  

In [ ]:
data[['pos_ols_1', 'pos_ols_2']].head()

In [ ]:
data[['pos_ols_1', 'pos_ols_2']] = np.where(
            data[['pos_ols_1', 'pos_ols_2']] > 0, 1, -1)  

In [ ]:
data['pos_ols_1'].value_counts()  

In [ ]:
data['pos_ols_2'].value_counts()  

In [ ]:
(data['pos_ols_1'].diff() != 0).sum()  

In [ ]:
(data['pos_ols_2'].diff() != 0).sum()  

In [ ]:
data['strat_ols_1'] = data['pos_ols_1'] * data['returns']

In [ ]:
data['strat_ols_2'] = data['pos_ols_2'] * data['returns']

In [ ]:
data[['returns', 'strat_ols_1', 'strat_ols_2']].sum().apply(np.exp)

In [ ]:
(data['direction'] == data['pos_ols_1']).value_counts()  

In [ ]:
(data['direction'] == data['pos_ols_2']).value_counts()  

In [ ]:
data[['returns', 'strat_ols_1', 'strat_ols_2']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

## Clustering

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
model = KMeans(n_clusters=2, random_state=0)  #  <1>

In [ ]:
model.fit(data[cols])

In [ ]:
data['pos_clus'] = model.predict(data[cols])

In [ ]:
data['pos_clus'] = np.where(data['pos_clus'] == 1, -1, 1)  

In [ ]:
data['pos_clus'].values

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(data[cols].iloc[:, 0], data[cols].iloc[:, 1],
            c=data['pos_clus'], cmap='coolwarm');

In [ ]:
data['strat_clus'] = data['pos_clus'] * data['returns']

In [ ]:
data[['returns', 'strat_clus']].sum().apply(np.exp)

In [ ]:
(data['direction'] == data['pos_clus']).value_counts()

In [ ]:
data[['returns', 'strat_clus']].cumsum().apply(np.exp).plot(figsize=(10, 6));

## Frequency Approach

In [ ]:
def create_bins(data, bins=[0]):
    global cols_bin
    cols_bin = []
    for col in cols:
        col_bin = col + '_bin'
        data[col_bin] = np.digitize(data[col], bins=bins)  
        cols_bin.append(col_bin)

In [ ]:
create_bins(data)

In [ ]:
data[cols_bin + ['direction']].head()  

In [ ]:
grouped = data.groupby(cols_bin + ['direction'])
grouped.size()  

In [ ]:
res = grouped['direction'].size().unstack(fill_value=0)  

In [ ]:
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: yellow' if v else '' for v in is_max]  

In [ ]:
res.style.apply(highlight_max, axis=1)  

In [ ]:
data['pos_freq'] = np.where(data[cols_bin].sum(axis=1) == 2, -1, 1)  

In [ ]:
(data['direction'] == data['pos_freq']).value_counts()

In [ ]:
data['strat_freq'] = data['pos_freq'] * data['returns']

In [ ]:
data[['returns', 'strat_freq']].sum().apply(np.exp)

In [ ]:
data[['returns', 'strat_freq']].cumsum().apply(np.exp).plot(figsize=(10, 6));

## Classification Algorithms

In [ ]:
from sklearn import linear_model
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

In [ ]:
C = 1

In [ ]:
models = {
    'log_reg': linear_model.LogisticRegression(C=C),
    'gauss_nb': GaussianNB(),
    'svm': SVC(C=C)
}

In [ ]:
def fit_models(data):  
    mfit = {model: models[model].fit(data[cols_bin], data['direction'])
            for model in models.keys()} 

In [ ]:
fit_models(data)

In [ ]:
def derive_positions(data):  
    for model in models.keys():
        data['pos_' + model] = models[model].predict(data[cols_bin])

In [ ]:
derive_positions(data)

In [ ]:
def evaluate_strats(data):  
    global sel
    sel = []
    for model in models.keys():
        col = 'strat_' + model 
        data[col] = data['pos_' + model] * data['returns']
        sel.append(col)
    sel.insert(0, 'returns')

In [ ]:
evaluate_strats(data)

In [ ]:
sel.insert(1, 'strat_freq')

In [ ]:
data[sel].sum().apply(np.exp)  

In [ ]:
data[sel].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
data = pd.DataFrame(raw[symbol])

In [ ]:
data['returns'] = np.log(data / data.shift(1))

In [ ]:
data['direction'] = np.sign(data['returns'])

In [ ]:
lags = 5  
create_lags(data)
data.dropna(inplace=True)

In [ ]:
create_bins(data)  
cols_bin

In [ ]:
data[cols_bin].head()

In [ ]:
data.dropna(inplace=True)

In [ ]:
fit_models(data)

In [ ]:
derive_positions(data)

In [ ]:
evaluate_strats(data)

In [ ]:
data[sel].sum().apply(np.exp)

In [ ]:
data[sel].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
mu = data['returns'].mean()  
v = data['returns'].std()  

In [ ]:
bins = [mu - v, mu, mu + v]  
bins  

In [ ]:
create_bins(data, bins)

In [ ]:
data[cols_bin].head()

In [ ]:
fit_models(data)

In [ ]:
derive_positions(data)

In [ ]:
evaluate_strats(data)

In [ ]:
data[sel].sum().apply(np.exp)

In [ ]:
data[sel].cumsum().apply(np.exp).plot(figsize=(10, 6));

### Sequential Train-Test Split

In [ ]:
split = int(len(data) * 0.5)

In [ ]:
train = data.iloc[:split].copy()  

In [ ]:
fit_models(train)  

In [ ]:
test = data.iloc[split:].copy()  

In [ ]:
derive_positions(test)  

In [ ]:
evaluate_strats(test)  

In [ ]:
test[sel].sum().apply(np.exp)

In [ ]:
test[sel].cumsum().apply(np.exp).plot(figsize=(10, 6));

### Randomized Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, test = train_test_split(data, test_size=0.5,
                               shuffle=True, random_state=100)

In [ ]:
train = train.copy().sort_index()  

In [ ]:
train[cols_bin].head()

In [ ]:
test = test.copy().sort_index()  

In [ ]:
fit_models(train)

In [ ]:
derive_positions(test)

In [ ]:
evaluate_strats(test)

In [ ]:
test[sel].sum().apply(np.exp)

In [ ]:
test[sel].cumsum().apply(np.exp).plot(figsize=(10, 6));

## Deep Neural Network

### DNN with scikit-learn

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
model = MLPClassifier(solver='lbfgs', alpha=1e-5,
                     hidden_layer_sizes=2 * [250], random_state=1)

In [ ]:
%time model.fit(data[cols_bin], data['direction'])

In [ ]:
data['pos_dnn_sk'] = model.predict(data[cols_bin])

In [ ]:
data['strat_dnn_sk'] = data['pos_dnn_sk'] * data['returns']

In [ ]:
data[['returns', 'strat_dnn_sk']].sum().apply(np.exp)

In [ ]:
data[['returns', 'strat_dnn_sk']].cumsum().apply(np.exp).plot(figsize=(10, 6));

In [ ]:
train, test = train_test_split(data, test_size=0.5, random_state=100)

In [ ]:
train = train.copy().sort_index()

In [ ]:
test = test.copy().sort_index()

In [ ]:
model = MLPClassifier(solver='lbfgs', alpha=1e-5, max_iter=500,
                     hidden_layer_sizes=3 * [500], random_state=1)  

In [ ]:
%time model.fit(train[cols_bin], train['direction'])

In [ ]:
test['pos_dnn_sk'] = model.predict(test[cols_bin])

In [ ]:
test['strat_dnn_sk'] = test['pos_dnn_sk'] * test['returns']

In [ ]:
test[['returns', 'strat_dnn_sk']].sum().apply(np.exp)

In [ ]:
test[['returns', 'strat_dnn_sk']].cumsum().apply(np.exp).plot(figsize=(10, 6));

### DNN with Keras & TensorFlow Backend

In [ ]:
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

In [ ]:
def create_model():
    np.random.seed(100)
    tf.random.set_seed(100)
    model = Sequential()
    model.add(Dense(16, activation='relu', input_dim=lags))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='rmsprop',
                  metrics=['accuracy'])
    return model

In [ ]:
data_ = (data - data.mean()) / data.std()
data['direction_'] = np.where(data['direction'] == 1, 1, 0)

In [ ]:
model = create_model()

In [ ]:
%%time
model.fit(data_[cols], data['direction_'],
          epochs=50, verbose=False)

In [ ]:
model.evaluate(data_[cols], data['direction_'])

In [ ]:
pred = np.where(model.predict(data_[cols]) > 0.5, 1, 0) 
pred[:10].flatten()

In [ ]:
data['pos_dnn_ke'] = np.where(pred > 0, 1, -1)  

In [ ]:
data['strat_dnn_ke'] = data['pos_dnn_ke'] * data['returns']

In [ ]:
data[['returns', 'strat_dnn_ke']].sum().apply(np.exp)

In [ ]:
data[['returns', 'strat_dnn_ke']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
mu, std = train.mean(), train.std()

In [ ]:
train_ = (train - mu) / mu.std()

In [ ]:
model = create_model()

In [ ]:
train['direction_'] = np.where(train['direction'] > 0, 1, 0)

In [ ]:
%%time
model.fit(train_[cols], train['direction_'],
          epochs=50, verbose=False)

In [ ]:
test_ = (test - mu) / std

In [ ]:
test['direction_'] = np.where(test['direction'] > 0, 1, 0)

In [ ]:
model.evaluate(test_[cols], test['direction_'])

In [ ]:
pred = np.where(model.predict(test_[cols]) > 0.5, 1, 0) 
pred[:10].flatten()

In [ ]:
test['pos_dnn_ke'] = np.where(pred > 0, 1, -1)

In [ ]:
test['strat_dnn_ke'] = test['pos_dnn_ke'] * test['returns']

In [ ]:
test[['returns', 'strat_dnn_sk', 'strat_dnn_ke']].sum().apply(np.exp)

In [ ]:
test[['returns', 'strat_dnn_sk', 'strat_dnn_ke']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>